# E3-H — Ablation contrôlée des informations locales et voisines

**Objectif** : distinguer l'apport de l'historique propre au dispositif de celui des historiques des voisins dans un modèle graphique.

**Précision essentielle :** ce notebook introduit une **architecture diagnostique à deux branches**, et **ne remplace pas** le GCN à deux couches de E3-F2A. Une ablation « voisin uniquement » réalisée avec un GCN à deux couches peut réintroduire l'information locale par des chemins de longueur deux. Ici, le modèle utilise explicitement une seule agrégation des voisins **sans boucle sur soi**, et des branches locales/voisines distinctes.

Configurations :
- **H1 — local** : branche locale seulement, aucune information voisine.
- **H2 — neighbors** : branche voisine seulement, sans caractéristique du nœud cible.
- **H3 — local+neighbors** : les deux branches, topologie opérationnelle.
- **H4 — local+shuffled** : les deux branches, topologie réorganisée à degré conservé.
- **H5 — local+neighbors permuted** : analyse *post-entraînement* de H3 ; permutation des trajectoires des voisins à l'inférence, en conservant les caractéristiques locales, sans réentraînement. Cette intervention introduit un décalage de distribution et n'est pas une estimation causale.

**Contrôle des comparaisons** : même jeu de données, mêmes horizons, mêmes graines, même largeur cachée fixée à partir de E3-F2A, même budget (200 époques, patience 25), mêmes cibles. L'architecture est identique pour H1–H4 ; seules les branches actives et/ou l'adjacence changent. Le nombre de paramètres effectivement entraînés varie selon les branches actives ; ce facteur est rapporté comme limite.

**Critère principal** : RMSE sur les 13 643 observations communes, par graine, et moyenne ± écart-type. Aucune sélection sur le test.


**Correctif v2 :** ajout de `K_NEIGHBORS=2`, paramètre de construction de la topologie qui manquait dans la version précédente.

In [ ]:
from pathlib import Path
import json, random, time, platform
import numpy as np
import pandas as pd
import networkx as nx
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT=Path.cwd().resolve()
REP=ROOT/'outputs/E3_aligned_representations'
COMMON=ROOT/'outputs/E3_common_evaluation'
GCN_OLD=ROOT/'outputs/E3_F2A_GCN_200epochs'
OUT=ROOT/'outputs/E3_H_LOCAL_NEIGHBOR_ABLATION'
HORIZONS=(15,30,60,120)
SEEDS=(42,123,2026)
SPLITS=('train','validation','test')
NODES=7; STEPS=5; FEATURES=3
K_NEIGHBORS=2  # Top-k utilisé dans E3-F2A pour la topologie opérationnelle
MAX_EPOCHS=200; PATIENCE=25
BATCH_SIZE=256; LR=1e-3; WEIGHT_DECAY=1e-5
MODES=('H1_local','H2_neighbors','H3_both','H4_both_shuffled')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert REP.is_dir() and COMMON.is_dir() and GCN_OLD.is_dir()
OUT.mkdir(parents=True,exist_ok=True)
common=pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp']=pd.to_datetime(common.timestamp,utc=True)
assert len(common)==91763
frozen=pd.read_csv(GCN_OLD/'05_selected_configs_and_training.csv')
WIDTH={}
for h in HORIZONS:
    widths=frozen.loc[frozen.horizon_min.eq(h),'hidden'].unique()
    assert len(widths)==1,(h,widths)
    WIDTH[h]=int(widths[0])
original=json.loads((GCN_OLD/'06_protocol.json').read_text(encoding='utf-8'))
print('Device:',device,'widths:',WIDTH)


In [ ]:
def load_horizon(h):
    folder=REP/f'h{h:03d}'
    nodes=pd.read_csv(folder/'01_graph_node_index.csv').sort_values('node_index')
    assert nodes['node_index'].tolist()==list(range(7))
    devices=nodes['uuid'].astype(str).tolist()
    data={}
    for split in SPLITS:
        X=np.load(folder/f'X_graph_{split}_scaled.npy').astype(np.float32)
        Xu=np.load(folder/f'X_graph_{split}_unscaled.npy',mmap_mode='r')
        y=np.load(folder/f'y_graph_{split}.npy').astype(np.float32)
        meta=pd.read_csv(folder/f'metadata_{split}.csv')
        meta['timestamp']=pd.to_datetime(meta['timestamp'],utc=True)
        meta['target_timestamp']=pd.to_datetime(meta['target_timestamp'],utc=True)
        idx=np.load(COMMON/f'indices_h{h:03d}_{split}.npy').astype(np.int64)
        assert X.shape==(len(y),5,7,3) and y.shape[1]==7
        assert len(meta)==y.size==len(idx)*0+7*len(y)
        assert len(idx)=={'train':64393,'validation':13727,'test':13643}[split]
        assert len(set(idx.tolist()))==len(idx) and idx.min()>=0 and idx.max()<y.size
        assert meta['uuid'].astype(str).tolist()==devices*len(y), 'Unexpected node order.'
        assert meta['split'].eq(split).all()
        assert (meta['target_timestamp']-meta['timestamp']).eq(pd.Timedelta(minutes=h)).all()
        assert np.allclose(meta['target'].to_numpy(),y.reshape(-1),rtol=0,atol=1e-5)
        assert np.isfinite(X).all() and np.isfinite(y).all()
        assert np.allclose(Xu.transpose(0,2,1,3).reshape(-1,5,3)[idx],
                           np.load(folder/f'X_sequence_{split}_unscaled.npy',mmap_mode='r')[idx],
                           rtol=0,atol=0)
        selected=meta.iloc[idx][['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        expected=common.loc[common.split.eq(split),['uuid','timestamp','split']].sort_values(['uuid','timestamp','split']).reset_index(drop=True)
        assert selected.equals(expected),f'Common keys differ h={h} split={split}'
        data[split]=dict(X=X,y=y,meta=meta,idx=idx,
                         persistence=np.asarray(Xu[:,-1,:,0],dtype=np.float64).reshape(-1))
        print(h,split,X.shape,'common nodes',len(idx))
    return data,devices

def build_topologies(train_X, h):
    # Multivariate train-only similarity: flatten all time snapshots and history
    # for each variable; standardized features are already train-only scaled.
    # Pearson correlation computed separately per variable, then averaged.
    v=train_X.transpose(2,0,1,3).reshape(7,-1,3)
    similarity=np.zeros((7,7),dtype=np.float64)
    for feature in range(3):
        z=v[:,:,feature]
        corr=np.corrcoef(z)
        assert np.isfinite(corr).all(), 'Constant training trajectory; inspect topology.'
        similarity+=corr/3
    np.fill_diagonal(similarity,-np.inf)
    edges=set()
    for i in range(7):
        order=sorted((j for j in range(7) if j!=i),
                     key=lambda j:(-similarity[i,j],j))
        for j in order[:K_NEIGHBORS]:
            edges.add(tuple(sorted((i,j))))
    g=nx.Graph();g.add_nodes_from(range(7));g.add_edges_from(sorted(edges))
    # Degree-preserving null topology. Fail if no genuine rewiring possible.
    shuffled=None
    for trial in range(100):
        cand=g.copy()
        try:
            nx.double_edge_swap(cand,nswap=2,max_tries=1000,seed=42+h+trial)
        except nx.NetworkXAlgorithmError:
            continue
        if set(cand.edges())!=set(g.edges()) and sorted(dict(cand.degree()).items())==sorted(dict(g.degree()).items()):
            shuffled=cand
            break
    assert shuffled is not None,'Could not construct a distinct degree-preserving shuffled topology.'
    assert dict(g.degree())==dict(shuffled.degree())
    def Ahat(graph):
        A=nx.to_numpy_array(graph,nodelist=list(range(7)),dtype=np.float32)
        A+=np.eye(7,dtype=np.float32)
        d=A.sum(axis=1)
        return torch.tensor(A/np.sqrt(d[:,None]*d[None,:]),dtype=torch.float32,device=device)
    topologies={'G_native':nx.empty_graph(7),
                'G_operational':g,
                'G_shuffled':shuffled}
    rows=[]
    for name,graph in topologies.items():
        for i,j in graph.edges():
            rows.append({'horizon_min':h,'topology':name,'node_i':i,'node_j':j})
    pd.DataFrame(rows,columns=['horizon_min','topology','node_i','node_j']).to_csv(
        OUT/f'h{h:03d}_topology_edges.csv',index=False)
    return {k:Ahat(v) for k,v in topologies.items()}, {
        k:sorted(tuple(sorted(e)) for e in v.edges()) for k,v in topologies.items()
    }


## Architecture à branches séparées

Pour chaque dispositif *i*, `self_i` correspond à ses 15 variables locales (5 instants × 3 mesures) et `neighbor_i` est la moyenne normalisée des 15 variables des voisins directs *j ≠ i*. Les matrices de voisinage ont une diagonale nulle ; une assertion vérifie qu'une perturbation de `self_i` ne peut pas affecter la prédiction du nœud *i* en mode H2. Les prédictions H1 ne dépendent d'aucun autre dispositif.

Les couches locales et voisines ont des poids distincts. Pour H1/H2, seule la branche pertinente est utilisée ; pour H3/H4, les deux branches sont additionnées. Le réseau n'effectue **aucune seconde propagation graphique**.


In [ ]:
class SeparatedNeighborModel(nn.Module):
    def __init__(self,hidden,mode):
        super().__init__()
        self.mode=mode
        self.self_fc=nn.Linear(STEPS*FEATURES,hidden)
        self.neighbor_fc=nn.Linear(STEPS*FEATURES,hidden)
        self.dropout=nn.Dropout(0.1)
        self.head=nn.Linear(hidden,1)

    def forward(self,x,A_neighbors):
        assert x.ndim==4 and tuple(x.shape[1:])==(STEPS,NODES,FEATURES)
        z=x.permute(0,2,1,3).contiguous().reshape(x.shape[0],NODES,STEPS*FEATURES)
        if self.mode=='H1_local':
            h=self.self_fc(z)
        elif self.mode=='H2_neighbors':
            h=self.neighbor_fc(torch.einsum('ij,bjf->bif',A_neighbors,z))
        elif self.mode in ('H3_both','H4_both_shuffled'):
            h=self.self_fc(z)+self.neighbor_fc(torch.einsum('ij,bjf->bif',A_neighbors,z))
        else:
            raise ValueError(self.mode)
        return self.head(self.dropout(torch.relu(h))).squeeze(-1)

def adjacency_no_self(edges):
    A=np.zeros((NODES,NODES),dtype=np.float32)
    for i,j in edges:
        A[i,j]=1;A[j,i]=1
    np.fill_diagonal(A,0)
    deg=A.sum(axis=1)
    assert np.all(deg>0),'Neighbor-only requires a neighbor for every device'
    A=A/deg[:,None]
    assert np.allclose(np.diag(A),0)
    assert np.allclose(A.sum(axis=1),1)
    return torch.tensor(A,device=device)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass

def loader(d,split,shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(d[split]['X']),
                                    torch.from_numpy(d[split]['y'])),
                      batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=0)

@torch.no_grad()
def predict(model,d,split,A):
    model.eval()
    return np.concatenate([model(x.to(device),A).cpu().numpy()
                           for x,_ in loader(d,split)],axis=0)

def train_model(d,hidden,mode,A,seed,checkpoint):
    seed_all(seed)
    model=SeparatedNeighborModel(hidden,mode).to(device)
    # Exclude inactive branch parameters from optimizer (no inadvertent weight decay).
    if mode=='H1_local':
        params=list(model.self_fc.parameters())+list(model.head.parameters())
    elif mode=='H2_neighbors':
        params=list(model.neighbor_fc.parameters())+list(model.head.parameters())
    else:
        params=list(model.parameters())
    optimizer=torch.optim.Adam(params,lr=LR,weight_decay=WEIGHT_DECAY)
    best=np.inf;stale=0;history=[]
    for epoch in range(1,MAX_EPOCHS+1):
        model.train();losses=[]
        for x,y in loader(d,'train',shuffle=True):
            x=x.to(device);y=y.to(device)
            optimizer.zero_grad()
            loss=nn.functional.mse_loss(model(x,A),y)
            loss.backward();optimizer.step()
            losses.append(float(loss.detach().cpu()))
        pv=predict(model,d,'validation',A)
        val_rmse=float(np.sqrt(np.mean((pv-d['validation']['y'])**2)))
        history.append({'epoch':epoch,'train_MSE':float(np.mean(losses)),
                        'validation_RMSE':val_rmse})
        if val_rmse<best-1e-7:
            best=val_rmse;stale=0
            torch.save(model.state_dict(),checkpoint)
        else:
            stale+=1
            if stale>=PATIENCE:break
    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True))
    return model,pd.DataFrame(history),best,sum(p.numel() for p in params)

def rmse(y,p):
    return float(np.sqrt(np.mean((np.asarray(y)-np.asarray(p))**2)))

# Structural non-leakage tests: change target's own history or other devices.
for mode in MODES:
    seed_all(42)
    A=adjacency_no_self([(i,(i+1)%NODES) for i in range(NODES)])
    net=SeparatedNeighborModel(16,mode).to(device).eval()
    x=torch.randn(2,STEPS,NODES,FEATURES,device=device)
    with torch.no_grad():
        base=net(x,A)
        altered_self=x.clone();altered_self[:,:,0,:]+=50
        changed_self=net(altered_self,A)
        altered_other=x.clone();altered_other[:,:,1:,:]+=50
        changed_other=net(altered_other,A)
    if mode=='H1_local':
        assert torch.allclose(base[:,0],changed_other[:,0],atol=1e-6)
    if mode=='H2_neighbors':
        assert torch.allclose(base[:,0],changed_self[:,0],atol=1e-6)
    print('No-leakage check passed:',mode)


## Entraînements et interventions sur les voisins

L'expérience comprend **48 entraînements** : 4 horizons × 4 configurations × 3 graines. La permutation H5 est évaluée **sans réentraînement** à partir des 12 modèles H3. Les trajectoires de voisins sont permutées entre les instantanés de test, en conservant le nœud cible et sa série locale ; les permutations sont **communes à tous les dispositifs** pour chaque instantané, afin de préserver la structure des voisins dans un instantané de provenance. Cette intervention mesure la sensibilité à la cohérence temporelle des voisins, mais ne prouve pas une causalité sémantique.

Pour éviter toute fuite, la sélection du checkpoint utilise exclusivement le split validation ; la permutation H5 n'influence aucun choix de modèle.


In [ ]:
records=[];predictions=[];trainings=[];edge_records=[]
for h in HORIZONS:
    print(f'\\n===== Horizon {h} min =====')
    d,devices=load_horizon(h)
    # Reuse exact E3-F2A topology construction, including its frozen shuffle.
    _,edges=build_topologies(d['train']['X'],h)
    for topology in ('G_operational','G_shuffled'):
        expected=sorted(tuple(sorted(map(int,e))) for e in original['topologies'][str(h)][topology])
        assert sorted(edges[topology])==expected,(h,topology,'topology mismatch')
        for i,j in edges[topology]:
            edge_records.append({'horizon_min':h,'topology':topology,'node_i':i,'node_j':j})
    A_op=adjacency_no_self(edges['G_operational'])
    A_sh=adjacency_no_self(edges['G_shuffled'])
    for mode in MODES:
        A=A_sh if mode=='H4_both_shuffled' else A_op
        for seed in SEEDS:
            t0=time.perf_counter()
            ckpt=OUT/f'h{h:03d}_{mode}_seed{seed}.pt'
            model,history,best,nparams=train_model(d,WIDTH[h],mode,A,seed,ckpt)
            history.to_csv(OUT/f'h{h:03d}_{mode}_seed{seed}_history.csv',index=False)
            trainings.append({'horizon_min':h,'mode':mode,'seed':seed,
                              'hidden':WIDTH[h],'active_parameters':nparams,
                              'best_validation_RMSE':best,
                              'best_epoch':int(history.loc[history.validation_RMSE.idxmin(),'epoch']),
                              'epochs_run':len(history),
                              'training_seconds':time.perf_counter()-t0})
            idx=d['test']['idx']
            y=d['test']['y'].reshape(-1)
            p=predict(model,d,'test',A).reshape(-1)
            meta=d['test']['meta'].iloc[idx].copy().reset_index(drop=True)
            assert len(meta)==len(idx)==13643
            pred=meta[['uuid','timestamp']].copy()
            pred['horizon_min']=h;pred['model']=mode;pred['seed']=seed
            pred['y_true']=y[idx];pred['y_pred']=p[idx]
            pred['persistence_pred']=d['test']['persistence'][idx]
            predictions.append(pred)
            records.append({'horizon_min':h,'mode':mode,'seed':seed,
                            'n':len(idx),'RMSE':rmse(y[idx],p[idx]),
                            'Persistence_RMSE':rmse(y[idx],d['test']['persistence'][idx]),
                            'evaluation':'common_keys'})
            if mode=='H3_both':
                # H5: shift *neighbor sources* across snapshots, while preserving target local features.
                # We do this by explicitly constructing local and neighbor tensors separately.
                X=d['test']['X']
                rng=np.random.default_rng(200000+h+seed)
                perm=rng.permutation(len(X))
                # Force non-identity and make temporal leakage explicit: only test data used.
                if np.any(perm==np.arange(len(X))):
                    perm=np.roll(perm,1)
                x_local=torch.from_numpy(X)
                x_neighbor=torch.from_numpy(X[perm].copy())
                all_preds=[]
                model.eval()
                with torch.no_grad():
                    for start in range(0,len(X),BATCH_SIZE):
                        loc=x_local[start:start+BATCH_SIZE].to(device)
                        nei=x_neighbor[start:start+BATCH_SIZE].to(device)
                        zl=loc.permute(0,2,1,3).contiguous().reshape(-1,NODES,15)
                        zn=nei.permute(0,2,1,3).contiguous().reshape(-1,NODES,15)
                        hloc=model.self_fc(zl)
                        hnei=model.neighbor_fc(torch.einsum('ij,bjf->bif',A_op,zn))
                        output=model.head(torch.relu(hloc+hnei)).squeeze(-1)
                        all_preds.append(output.cpu().numpy())
                p5=np.concatenate(all_preds).reshape(-1)
                pred5=pred.copy()
                pred5['model']='H5_both_neighbors_permuted'
                pred5['y_pred']=p5[idx]
                predictions.append(pred5)
                records.append({'horizon_min':h,'mode':'H5_both_neighbors_permuted',
                                'seed':seed,'n':len(idx),'RMSE':rmse(y[idx],p5[idx]),
                                'Persistence_RMSE':rmse(y[idx],d['test']['persistence'][idx]),
                                'evaluation':'posthoc_test_neighbor_permutation'})
            print('Finished',h,mode,seed,'test RMSE',round(records[-1]['RMSE'],5))
            del model
            if torch.cuda.is_available():torch.cuda.empty_cache()

pred_df=pd.concat(predictions,ignore_index=True)
metric_df=pd.DataFrame(records)
train_df=pd.DataFrame(trainings)
edge_df=pd.DataFrame(edge_records)
assert len(train_df)==48
assert len(metric_df)==60
assert len(pred_df)==60*13643
assert not pred_df.duplicated(['horizon_min','model','seed','uuid','timestamp']).any()
summary=(metric_df.groupby(['horizon_min','mode'],as_index=False)
         .agg(n_seeds=('seed','nunique'),RMSE_mean=('RMSE','mean'),RMSE_SD=('RMSE','std'),
              Persistence_RMSE=('Persistence_RMSE','mean')))
assert summary.n_seeds.eq(3).all()
summary.to_csv(OUT/'00_PRIMARY_H_ablation_mean_sd.csv',index=False)
metric_df.to_csv(OUT/'01_H_per_seed_metrics.csv',index=False)
pred_df.to_csv(OUT/'02_H_common_test_predictions.csv',index=False)
train_df.to_csv(OUT/'03_H_training_diagnostics.csv',index=False)
edge_df.to_csv(OUT/'04_H_edges.csv',index=False)
protocol={
 'experiment':'E3-H local versus neighbor separated-branch diagnostic',
 'architecture':'separate linear local and one-hop neighbor branches, relu dropout head',
 'comparability':'new diagnostic architecture; not the two-layer GCN of E3-F2A',
 'input_shape':'(snapshots,5,7,3)',
 'neighbor_aggregation':'row-normalized adjacency with zero diagonal',
 'frozen_hidden_by_horizon':WIDTH,
 'modes':list(MODES)+['H5_both_neighbors_permuted'],
 'H5':'posthoc test-only permutation of neighbor source snapshots; no retraining',
 'horizons_minutes':list(HORIZONS),'seeds':list(SEEDS),
 'max_epochs':MAX_EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE,
 'learning_rate':LR,'weight_decay':WEIGHT_DECAY,
 'test_samples_per_horizon_seed_mode':13643,
 'limitations':['H1/H2 have fewer active trainable parameters than H3/H4',
                'neighbor-only can exploit correlations but not target local history',
                'H5 is distribution shift, not causal proof',
                'only one building/test period and three seeds',
                'operational topology is correlation-derived, not a direct semantic-edge ablation'],
 'python':platform.python_version(),'torch':torch.__version__,
 'networkx':nx.__version__}
(OUT/'05_H_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print(summary.to_string(index=False))
print('Saved:',OUT)


## Contrôles et interprétation

- **H1 vs H2** : utilité prédictive relative des seules données locales et des seules données voisines. La comparaison n'est pas parfaitement contrôlée en termes d'information disponible.
- **H3 vs H1** : gain potentiel de l'information voisine *en plus* de l'historique local. Ce contraste reste influencé par la capacité du modèle à deux branches.
- **H3 vs H4** : sensibilité à l'organisation des relations avec la même architecture et le même budget.
- **H3 vs H5** : dépendance du modèle entraîné à la cohérence temporelle des voisins, sans réentraînement (attention au décalage de distribution).
- **Référence** : comparer à la persistance, qui reste très performante.
- **Étape statistique** : calculer les IC appariés par blocs temporels pour les différences de RMSE sur les prédictions `02_H_common_test_predictions.csv`, en tenant compte des trois graines.

**Ne pas écrire que l'ablation démontre la valeur causale de Brick/IFC/SOSA.** Elle isole les canaux d'information et l'effet de la topologie dans une architecture diagnostique.
